In [1]:
!uv pip list

Package                                  Version     Editable project location
---------------------------------------- ----------- ----------------------------
aiohappyeyeballs                         2.6.1
aiohttp                                  3.12.13
aiohttp-retry                            2.9.1
aiosignal                                1.3.2
annotated-types                          0.7.0
anthropic                                0.54.0
anyio                                    4.9.0
asttokens                                3.0.0
asyncstdlib-fw                           3.13.2
attrs                                    25.3.0
backoff                                  2.2.1
bcrypt                                   4.3.0
betterproto-fw                           2.0.3
blockbuster                              1.5.24
build                                    1.2.2.post1
cachetools                               5.5.2
certifi                                  2025.6.15
cffi                    

Using Python 3.12.11 environment at: C:\cursor\langgraph_baseline\.venv


# RAG 시스템 설계
하나하나 기능 단위로 시작

#### 0. env - API Load

In [15]:
from dotenv import load_dotenv

load_dotenv()

True

## 1. Document
- custom document 생성

---
##### Error & solve
- 원래는 jsonloader를 사용해서 바로 만드려고 했는데, jonsl파일이 통일된 구조도 안되고 곳곳에 값도 안차서 오류가 나서 실패
-> 그래서 custom loader로 만듬


###### 0. JSONL 파악

In [16]:
import json

file_path = r'C:\cursor\langgraph_baseline\oliveyoung\final_merged_products.jsonl'
count = 0
max_count = 2 # 확인하고 싶은 객체 수

with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        if count >= max_count:
            break # max_count 만큼 확인했으면 반복문 종료

        try:
            obj = json.loads(line)
            print(f"--- 객체 {count + 1} ---")
            print(obj)
            count += 1
        except json.JSONDecodeError:
            print(f"다음 줄에서 JSON 파싱 오류 발생: {line.strip()}")

--- 객체 1 ---
{'product_name': '[7.12 하루특가/7월 올영픽]라로슈포제 시카플라스트 멀티 리페어 크림 B5 100ml 기획 (멜라B3 세럼 3ml)', 'brand': '라로슈포제', 'category_path': ['스킨케어', '크림', '크림'], 'original_price': 50000, '용량': '100ml+3ml', '제조/판매업자': '프랑스 라로슈포제사 / 엘오케이 유한회사', '전성분': '시카플라스트 멀티 리페어 크림\n정제수 다이카프릴릴에터 판테놀 글리세린 펜틸렌글라이콜 폴리글리세릴-6다이스테아레이트 프로판다이올 세틸에스터 호호바에스터 베헤닐알코올 그린와틀꽃왁스 해바라기씨왁스 병풀잎추출물 야콘뿌리즙 솔비탄올리에이트 징크글루코네이트 마데카소사이드 세테아릴아이소노나노에이트 망가니즈글루코네이트 아이소헥사데칸 알파-글루칸올리고사카라이드 소듐하이알루로네이트 소듐스테아로일글루타메이트 아데노신 만노오스 카퍼글루코네이트 하이드록시아세토페논 하이드록시프로필스타치포스페이트 카프릴로일살리실릭애씨드 비트레오스실라발효물 시트릭애씨드 트라이소듐에틸렌다이아민다이석시네이트 락토바실러스 말토덱스트린 폴리글리세린-3 폴리글리세릴-3비즈왁스 폴리솔베이트80 아크릴아마이드/소듐아크릴로일다이메틸타우레이트코폴리머 세틸알코올 토코페롤\n\n멜라B3 세럼\n정제수 다이메티콘 나이아신아마이드 글리세린 프로필렌글라이콜 폴리실리콘-11 실리카 비스-피이지/피피지-16/16피이지/피피지-16/16다이메티콘 레인보우랙추출물 2-머캅토니코티노일글라이신 피이지-20메틸글루코오스세스퀴스테아레이트 소듐하이알루로네이트 소듐하이드록사이드 소듐티오설페이트 카르노신 폴록사머338 암모늄폴리아크릴로일다이메틸타우레이트 다이포타슘글리시리제이트 카프릴릭/카프릭트라이글리세라이드 카프릴로일살리실릭애씨드 카프릴릴글라이콜 시트릭애씨드 트라이소듐에틸렌다이아민다이석시네이트 잔탄검 펜틸렌글라이콜 옥틸도데칸올 레티닐팔미테이트 토코페롤 펜타에리스리틸테트라-다이-t-부틸하이드록시하이드로신나메이트 페녹

### 1. Custom Document

document 만드는 방법... 쉽다!

In [17]:
from langchain.schema import Document

doc = Document(
    page_content="Hello!",
    metadata={"source": "test"}
)

doc

Document(metadata={'source': 'test'}, page_content='Hello!')

In [2]:
import json
from langchain_core.documents import Document

# 변환할 JSONL 파일 경로를 지정하세요.
# 집-경로
# jsonl_file_path = 'C:\\cursor\\DealMakers\\langgraph_baseline\\oliveyoung\\final_merged_products.jsonl' 
# 회사 경로
jsonl_file_path =  "C:\\cursor\\langgraph_baseline\\oliveyoung\\final_merged_products.jsonl"
documents = []

try:
    # 파일을 한 줄씩 읽어 메모리 문제를 방지합니다.
    with open(jsonl_file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            # 비어있는 줄은 건너뜁니다.
            if not line.strip():
                continue
            
            try:
                # 한 줄을 JSON 객체(파이썬 딕셔너리)로 변환합니다.
                obj = json.loads(line)

                # 1. page_content 생성 (핵심 텍스트 내용)
                page_content = obj.get('pdf_summary', '')
                
                # 2. metadata 생성 (부가 정보)
                # .get() 메소드를 사용하면 특정 키가 없어도 오류 없이 안전하게 값을 가져올 수 있습니다.
                metadata = {
                    'name': obj.get('product_name') or '',
                    'brand': obj.get('brand') or '',
                    'category': obj.get('category_path') or '',
                    'price': obj.get('original_price') or 0,
                    'volume': obj.get('용량') or '',
                    'manufacturer': obj.get('제조/판매업자') or '',
                    'review': obj.get('review_summary') or '',
                    'ingredients': obj.get('전성분') or ''# 전성분도 메타데이터로 관리하여 필터링에 활용 가능
                }

                # 3. LangChain Document 객체 생성
                doc = Document(page_content=page_content, metadata=metadata)
                
                # 생성된 객체를 리스트에 추가
                documents.append(doc)

            except json.JSONDecodeError:
                print(f"경고: {i+1}번째 줄에서 JSON 파싱 오류가 발생했습니다. 해당 줄을 건너뜁니다.")
            except Exception as e:
                print(f"경고: {i+1}번째 줄 처리 중 오류 발생: {e}")

    # 변환 결과 확인
    print(f"성공적으로 총 {len(documents)}개의 Document 객체를 생성했습니다.")
    
    # 첫 번째 Document 객체가 어떻게 만들어졌는지 샘플로 확인
    if documents:
        print("\n--- 첫 번째 Document 객체 샘플 ---")
        print(documents[0])
        
        print("\n--- 첫 번째 Document의 메타데이터 ---")
        print(documents[0].metadata)


except FileNotFoundError:
    print(f"오류: 파일을 찾을 수 없습니다 - {jsonl_file_path}")
except Exception as e:
    print(f"알 수 없는 오류가 발생했습니다: {e}")

경고: 4번째 줄 처리 중 오류 발생: 1 validation error for Document
page_content
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type
경고: 16번째 줄 처리 중 오류 발생: 1 validation error for Document
page_content
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type
경고: 21번째 줄 처리 중 오류 발생: 1 validation error for Document
page_content
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type
경고: 42번째 줄 처리 중 오류 발생: 1 validation error for Document
page_content
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type
경고: 77번째 줄 처리 중 오류 발생: 1 validation error

여기서는 `get("key")`에서 `~.get("key") or ''`으로 최종으로 됐는데,

전자에서는 None값을 대처하지 못했어서, 값이 없더라도 어떻게든 값을 채워넣어야

밑에 과정들에서 오류가 안남

같은 구조 유지가 중요!

---
애초에 ... 데이터를 이쁘게 모으면 참 좋겠다..

### 2. Chunk Splitter

In [20]:
# Cell 8: 텍스트 분할 테스트
"""
텍스트를 어떻게 나눌지 실험해보자
"""

from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 분할기 생성
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,  # 각 청크의 최대 크기
    chunk_overlap=0,  # 청크 간 겹치는 부분
    separators=["\n\n", "\n", ". ", " "]  # 분할 우선순위
)

# 첫 번째 문서로 테스트
test_content = documents[0].page_content
chunks = child_splitter.split_text(test_content)

print(f"원본 문서: {len(test_content)} 글자")
print(f"분할 결과: {len(chunks)}개 청크\n")

# 처음 2개 청크 확인
for i, chunk in enumerate(chunks[:2]):
    print(f"청크 {i+1}:")
    print(chunk)
    print("-" * 30)

print(f"Child 청크 크기: {child_splitter._chunk_size}")
print(f"청크 겹침: {child_splitter._chunk_overlap}")

원본 문서: 5137 글자
분할 결과: 77개 청크

청크 1:
# 고객 중심 컨셉 분석 리포트: 라로슈포제 시카플라스트 멀티 리페어 크림

---

## 1. 제품 기본 정보 (Product Identity)
------------------------------
청크 2:
*   **제품명**: 시카플라스트 멀티 리페어 크림 (Cicaplast Multi Repair Cream)
*   **브랜드**: 라로슈포제 (LA ROCHE POSAY)
------------------------------
Child 청크 크기: 100
청크 겹침: 0


여기서도 처음엔 `chunk_size`가 500이였는데,

pinecone에 생각보다 작아서(4mb?) 청크 분할을 적게적게 하다보니,

여기까지 내려옴

---
- overlap은 어차피 전체 문서를 참조할 것이기 때문에 전혀 상관이 없었음

In [18]:
len(documents)

841

### 3. Embedding & Vector Store - Pinecone

##### Error : Document가 너무 길어서 처리 임베딩에서 처리 불가 - 청킹 필수

#### 처음 세팅!
한 번만 하면, 그 이후론 필요없음
- 사실, 조건문이라 실행해도 상관없음 확인차로 돌리긴해야함

In [ ]:
from pinecone import Pinecone, ServerlessSpec


# 인덱스 만들기
pc = Pinecone()

index_name = "oliveyoung-rag"  # change if desired

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

c:\cursor\langgraph_baseline\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 4. Retriever

In [ ]:
# Cell 9: ParentDocumentRetriever 소개
"""
## Step 5: ParentDocumentRetriever 사용하기

우리가 원하는 것:
1. 작은 청크로 검색 (정확도 ↑)
2. 전체 문서 반환 (맥락 유지)

ParentDocumentRetriever가 정확히 이걸 해준다!
"""
from langchain_pinecone import PineconeVectorStore
from langchain.retrievers import ParentDocumentRetriever
# from langchain.storage import InMemoryStore
from langchain.storage import LocalFileStore, create_kv_docstore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

print("🎯 ParentDocumentRetriever의 작동 방식:")
print("1. Parent 문서를 작은 Child 청크로 분할")
print("2. Child 청크를 벡터DB에 저장 (검색용)")
print("3. Parent 문서는 별도 저장소에 보관")
print("4. 검색 시: Child 검색 → Parent 반환")

# Cell 10: ParentDocumentRetriever 설정
"""
이제 실제로 설정해보자!
"""

# 1. 임베딩 모델 준비
embeddings = OpenAIEmbeddings()

# 2. 벡터스토어 준비 (Child 청크 저장용)
index = pc.Index(index_name)
vectorstore = PineconeVectorStore(index=index, embedding=embeddings)

# 3. Parent 문서 저장소
# docstore = InMemoryStore()
store = LocalFileStore(r"C:\cursor\langgraph_baseline\oliveyoung\pinecone_db")
docstore = create_kv_docstore(store)


# 4. ParentDocumentRetriever 생성
parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=docstore, # 래핑된 docstore를 전달
    child_splitter=child_splitter,
    child_metadata_fields=["name", "category", "price", "volume"]
)

🎯 ParentDocumentRetriever의 작동 방식:
1. Parent 문서를 작은 Child 청크로 분할
2. Child 청크를 벡터DB에 저장 (검색용)
3. Parent 문서는 별도 저장소에 보관
4. 검색 시: Child 검색 → Parent 반환
✅ ParentDocumentRetriever 설정 완료!
  - Child 청크 크기: 100
  - 청크 겹침: 0


#### 문서 추가

In [6]:
# 한 번에 처리할 문서의 수를 정합니다 (서버나 문서 크기에 따라 조절).
# 예를 들어 50개 또는 100개로 시작해 보세요.
doc_batch_size = 1

# 전체 문서를 정해진 배치 크기만큼 잘라 루프를 실행합니다.
for i in range(0, len(documents), doc_batch_size):
    # documents 리스트에서 일부만 잘라 batch_docs를 만듭니다.
    batch_docs = documents[i : i + doc_batch_size]
    
    # 잘라낸 작은 묶음(batch)만 retriever에 추가합니다.
    parent_retriever.add_documents(batch_docs)
    
    # 진행 상황을 확인하기 위한 로그 (선택 사항)
    print(f"문서 {i + len(batch_docs)} / {len(documents)} 처리 완료")

print("모든 문서 추가 완료!")

# 저장된 내용 확인
print(f"\n📊 저장 통계:")
print(f"  - Parent 문서 수: {len(list(docstore.yield_keys()))}")
# print(f"  - Child 청크 수: {vectorstore._collection.count()}")

# Parent 문서 ID 확인
print("\n저장된 Parent 문서 ID:")
for key in list(docstore.yield_keys())[:5]:  # 처음 5개만
    print(f"  - {key}")

문서 1 / 841 처리 완료
문서 2 / 841 처리 완료
문서 3 / 841 처리 완료
문서 4 / 841 처리 완료
문서 5 / 841 처리 완료
문서 6 / 841 처리 완료
문서 7 / 841 처리 완료
문서 8 / 841 처리 완료
문서 9 / 841 처리 완료
문서 10 / 841 처리 완료
문서 11 / 841 처리 완료
문서 12 / 841 처리 완료
문서 13 / 841 처리 완료
문서 14 / 841 처리 완료
문서 15 / 841 처리 완료
문서 16 / 841 처리 완료
문서 17 / 841 처리 완료
문서 18 / 841 처리 완료
문서 19 / 841 처리 완료
문서 20 / 841 처리 완료
문서 21 / 841 처리 완료
문서 22 / 841 처리 완료
문서 23 / 841 처리 완료
문서 24 / 841 처리 완료
문서 25 / 841 처리 완료
문서 26 / 841 처리 완료
문서 27 / 841 처리 완료
문서 28 / 841 처리 완료
문서 29 / 841 처리 완료
문서 30 / 841 처리 완료
문서 31 / 841 처리 완료
문서 32 / 841 처리 완료
문서 33 / 841 처리 완료
문서 34 / 841 처리 완료
문서 35 / 841 처리 완료
문서 36 / 841 처리 완료
문서 37 / 841 처리 완료
문서 38 / 841 처리 완료
문서 39 / 841 처리 완료
문서 40 / 841 처리 완료
문서 41 / 841 처리 완료
문서 42 / 841 처리 완료
문서 43 / 841 처리 완료
문서 44 / 841 처리 완료
문서 45 / 841 처리 완료
문서 46 / 841 처리 완료
문서 47 / 841 처리 완료
문서 48 / 841 처리 완료
문서 49 / 841 처리 완료
문서 50 / 841 처리 완료
문서 51 / 841 처리 완료
문서 52 / 841 처리 완료
문서 53 / 841 처리 완료
문서 54 / 841 처리 완료
문서 55 / 841 처리 완료
문서 56 / 841 처리 완료
문

#### RAG 질문

In [ ]:
query = "피부 보습"

# 추후 LLM이 자동으로 판단해서 넣을 수 있도록
category = "스킨케어"

filter_condition = {"category": {"$in": ["로션"]}}

# vectorstore에서 직접 검색
child_docs = vectorstore.similarity_search(
	query,
	k=3,
	filter=filter_condition
)

# parent document ID를 추출하여 parent documents 가져오기
parent_ids = []
for doc in child_docs:
	if hasattr(doc, 'metadata') and 'doc_id' in doc.metadata:
		parent_ids.append(doc.metadata['doc_id'])
 
# 중복 제거
parent_ids = list(set(parent_ids))

# docstore에서 parent documents 가져오기
filtered_docs = []
for parent_id in parent_ids:
	parent_doc = docstore.mget([parent_id])[0]
	if parent_doc:
		filtered_docs.append(parent_doc)
		
# 문서 확인
# child_docs
# filtered_docs

In [ ]:
from rich import print as rprint

rprint(child_docs)


[
    Document(
        id='3df252ca-1666-4607-ab0e-dd142e0920e6',
        metadata={
            'category': ['스킨케어', '로션', '올인원'],
            'doc_id': 'c9343eb4-12c3-416a-adf4-c2e4de73b955',
            'name': '[RENEW]아이디얼포맨 올인원 3종 (프레시/시카/선)',
            'price': 26000.0,
            'volume': '시카올인원 150mL / 프레시올인원 150ml / 선 올인원 140ml'
        },
        page_content='피부 장벽 강화.'
    ),
    Document(
        id='46193bae-79fe-4ce9-86ef-0d82dbdab28b',
        metadata={
            'category': ['스킨케어', '로션', '올인원'],
            'doc_id': '0aa54bc9-2f69-4052-8710-21b7899cd76f',
            'name': '[개기름/지성] 닥터지 레드 블레미쉬 포 맨 올인원 오일컷 로션 150ml 3종 택1 (단품/증정 기획)',
            'price': 32000.0,
            'volume': '150ml'
        },
        page_content='매끈하고 건강한 피부로 완성합니다.'
    ),
    Document(
        id='2364a724-3891-4a0b-9d1c-903816a82f41',
        metadata={
            'category': ['스킨케어', '로션', '올인원'],
            'doc_id': '2936d78f-c013-4a7c-b9ce-affc46f7463c',
            'name': '[단독기획]토리든 다이브인 포맨 저분자 히알루론산 올인원 200g 단품/기획 (+클렌징폼 30ml/올인원 
20g)',
            'price': 24000.0,
            'volume': '200 g / 7.05 oz.'
        },
        page_content='*   **판테놀**: 피부 진정과 보습에 도움을 주어 수분 진정 시너지 효과를 제공합니다'
    )
]

In [14]:
rprint(filtered_docs)

[
    Document(
        metadata={
            'name': '[RENEW]아이디얼포맨 올인원 3종 (프레시/시카/선)',
            'brand': '아이디얼포맨',
            'category': ['스킨케어', '로션', '올인원'],
            'price': 26000,
            'volume': '시카올인원 150mL / 프레시올인원 150ml / 선 올인원 140ml',
            'manufacturer': '화장품 책임판매업자: 씨제이올리브영 / 화장품 제조업자: 한국콜마',
            'review': '## [RENEW] 아이디얼포맨 올인원 3종 (프레시/시카/선) 제품 리뷰 분석 및 성장 전략 
보고서\n\n### 핵심 요약 (Executive Summary)\n\n본 보고서는 \'[RENEW] 아이디얼포맨 올인원 3종 (프레시/시카/선)\' 
제품에 대한 사용자 리뷰 데이터를 종합적으로 분석하여 제품의 문제점을 진단하고, 이를 극복하여 지속 가능한 성장을 
도모할 수 있는 실행 가능한 전략을 제안합니다. 리뷰 분석 결과, 제품의 핵심적인 문제점은 **① 부족한 보습력 및 건조함 
유발, ② 사용 편의성을 저해하는 용기 디자인(뚜껑 개폐 어려움), ③ 선크림 기능의 미흡함**으로 나타났습니다. 특히, 이 
문제는 복합성 및 건성 피부 타입 사용자에게서 더 두드러지게 나타나고 있으며, 일부는 따가움이나 트러블을 
경험했습니다. 이러한 문제점을 해결하고 긍정적인 사용자 경험을 제공하기 위해, R&D 차원에서의 보습력 강화 및 성분 
개선, 사용 편의성을 높이는 용기 디자인 개선, 그리고 선케어 기능 명확화 및 강화가 시급합니다. 또한, 명확한 타겟 
고객층 설정과 그에 맞는 마케팅 및 커뮤니케이션 전략 수립을 통해 브랜드 신뢰도를 회복하고 새로운 시장 기회를 
포착해야 합니다.\n\n### 상세 분석 및 우선순위 (Detailed Analysis & Prioritization)\n\n**1. 문제점 체계적 분류 및 
근본 원인 분석**\n\n*   **제형 및 사용감 영역**:\n    *   **부족한 보습력 및 건조함 유발**: 다수의 리뷰에서 제품 
사용 후 건조함을 느끼거나 각질이 일어난다는 피드백이 있었습니다. 특히 복합성 및 건성 피부 타입 사용자는 보습력이 
부족하여 추가적인 보습 제품을 사용해야 한다고 언급했습니다. 이는 제품의 보습 성분 함량 부족 또는 특정 피부 타입에 
대한 보습력 강화가 미흡한 것이 근본 원인으로 추정됩니다. (언급 빈도: 높음, 비즈니스 임팩트: 높음, 근본 원인 가설: 
보습 성분 함량 부족 또는 포뮬러의 보습력 강화 미흡)\n    *   **퍽퍽한 마무리감**: 일부 사용자는 제품이 퍽퍽하게 
느껴지거나 뻑뻑하다는 의견을 남겼습니다. 이는 제형의 점도가 높거나 특정 성분의 특성으로 인해 발생할 수 있으며, 
결과적으로 다른 제품과 섞어 사용해야 하는 불편함을 야기합니다. (언급 빈도: 중간, 비즈니스 임팩트: 중간, 근본 원인 
가설: 제형의 점도 또는 특정 성분의 사용감 문제)\n    *   **눈 시림**: 한 리뷰에서 제품 사용 시 눈이 시렵다는 의견이
있었습니다. 이는 특정 성분이 눈 주변에 자극을 줄 수 있음을 시사하며, 포뮬러 구성이나 성분 선택에 대한 재검토가 
필요합니다. (언급 빈도: 낮음, 비즈니스 임팩트: 높음, 근본 원인 가설: 특정 성분의 눈 자극 가능성)\n\n*   **성능 및 
효과 영역**:\n    *   **선크림 기능 미흡**: 선크림 제품에서 선크림으로서의 효과가 부족하다는 의견이 있었습니다. 
다른 올인원 제품 대비 선크림 효과가 미미하며, 물로만 세정하기 어렵다는 피드백은 제품이 단순히 로션+선크림 질감으로 
느껴진다는 점을 뒷받침합니다. 이는 제품이 SPF 지수나 자외선 차단 성분 함량 등에서 기대치를 충족시키지 못함을 
의미합니다. (언급 빈도: 높음, 비즈니스 임팩트: 높음, 근본 원인 가설: 자외선 차단 성분 함량 부족 또는 선케어 기능의 
전반적인 약화)\n    *   **개기름 관리 효과 불명확**: 일부 지성 피부 사용자는 개기름을 잡는 데 효과적이라고 
언급했으나, 이로 인해 피부가 건조해지고 트러블이 발생했다는 상반된 의견도 존재합니다. 이는 제품이 유분 컨트롤 
기능과 보습 기능을 동시에 균형 있게 제공하지 못함을 시사합니다. (언급 빈도: 중간, 비즈니스 임팩트: 중간, 근본 원인 
가설: 유수분 밸런스 조절 기능의 불균형)\n\n*   **신체적/생리적 반응 영역**:\n    *   **피부 트러블 유발 및 
따가움**: 건조함으로 인한 트러블 발생 및 피부 따가움에 대한 언급이 있었습니다. 특히 민감성 피부 사용자에게서 이러한
문제가 나타날 가능성이 있으며, 이는 특정 성분의 자극 가능성을 시사합니다. (언급 빈도: 낮음, 비즈니스 임팩트: 높음, 
근본 원인 가설: 특정 성분의 자극 가능성 또는 포뮬러의 민감성 고려 부족)\n\n*   **포장 및 사용 편의성 영역**:\n    *
**용기 디자인 불편함**: 펌핑식이 아닌 플립탑 캡 방식의 용기 디자인에 대한 불편함이 여러 리뷰에서 공통적으로 
지적되었습니다. 뚜껑이 잘 열리지 않거나 닫히지 않는다는 의견은 사용자 경험을 심각하게 저해하는 요인으로 작용합니다.
(언급 빈도: 높음, 비즈니스 임팩트: 높음, 근본 원인 가설: 용기 구조 설계의 오류 또는 재질 문제)\n\n*   **가치 및 
전반적 인식 영역**:\n    *   **기대 불일치**: 올인원 제품의 편리성에 대한 기대와 실제 사용감(건조함, 용기 불편함 
등) 사이의 괴리가 일부 사용자에게 실망감을 안겨주었습니다. 전반적으로 \'무난하다\'는 평가가 많지만, 압도적인 
만족감을 주는 포인트는 부족해 보입니다. (언급 빈도: 높음, 비즈니스 임팩트: 중간, 근본 원인 가설: 제품의 핵심 
가치(편리함) 외 기능적 만족도 부족)\n\n**2. \'도움돼요\' 지표를 활용한 핵심 문제 우선순위화**\n\n리뷰 데이터에서 
\'도움돼요\' 지표가 0으로만 기록되어 있어 커뮤니티 공감도를 직접적으로 측정하기 어렵습니다. 따라서, **언급 빈도와 
문제의 심각성(비즈니스 임팩트)을 종합적으로 고려하여 우선순위를 설정**합니다.\n\n1.  **[Top 1] 부족한 보습력 및 
건조함 유발**: 언급 빈도가 높고, 건조함 및 각질 유발이라는 직접적인 피부 불만으로 이어져 사용자의 재구매 의사를 
저해할 수 있는 치명적인 문제입니다. 특히 복합성 및 건성 피부 사용자에게 큰 영향을 미칩니다.\n2.  **[Top 2] 사용 
편의성을 저해하는 용기 디자인**: 뚜껑 개폐의 어려움은 매일 반복되는 사용 경험에서 큰 불편함을 초래하며, 이는 제품에
대한 부정적인 인식을 형성하는 데 결정적인 역할을 합니다. 언급 빈도도 높고, 디자인 개선 요구가 명확하여 시급한 
개선이 필요합니다.\n3.  **[Top 3] 선크림 기능의 미흡함**: \'선\' 제품 라인이 포함되어 있음에도 불구하고 
선크림으로서의 기능이 부족하다는 점은 제품의 핵심 가치를 훼손하며, 소비자의 기대에 부응하지 못하는 명확한 실패 
요인입니다. 이는 제품의 전반적인 성능에 대한 신뢰도를 하락시킵니다.\n\n이 외에도 눈 시림, 피부 트러블, 따가움 등은 
언급 빈도는 낮지만 사용자에게 심각한 부정적 경험을 제공할 수 있는 **숨겨진 치명적 문제점**으로 간주되어 면밀한 
검토가 필요합니다.\n\n### 문제점과 핵심 타겟 그룹 연결 (Link Issues to Key Target Segments)\n\n분석 결과, **복합성 
피부 타입 사용자**에게서 가장 많은 피드백이 집중되었습니다. 특히 이들은 다음과 같은 문제점을 더 민감하게 느끼는 
것으로 나타났습니다.\n\n*   **부족한 보습력 및 건조함 유발**: 복합성 피부는 부위별 유수분 밸런스가 다른 특성상, 
보습력이 부족하면 건조한 부위에 각질이 일어나거나 당기는 느낌을 받을 수 있습니다. 리뷰에서 "완전 지성이신 분 아니면
비추합니다", "다른 수분크림과 섞어 사

In [18]:
from pathlib import Path
from langchain.storage import LocalFileStore

path = Path.cwd() / "data"
print(path)

# 부모 문서를 위한 Document Store (메모리 내 저장소 사용)
store = LocalFileStore(path)

c:\cursor\langgraph_baseline\oliveyoung\data


In [59]:
fs_path
chroma_db_path

'C:\\cursor\\langgraph_baseline\\oliveyoung\\chroma_db'